# 101 — Re-ranking y filtros de evidencia

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Bi-encoder**: codifica consulta y documento por separado; relevancia = coseno entre
vectores. Pre-computable e indexable → 1.ª etapa (recall), pero sin interacción q-d.

**Cross-encoder** (arXiv:1901.04085): una sola entrada `[CLS] q [SEP] d` con atención
completa entre todos los tokens → score de relevancia preciso, pero nada pre-computable:
una pasada del modelo por cada par. Solo viable sobre decenas de candidatos → 2.ª etapa.

**Pipeline retrieve-then-rerank**: recuperar amplio y barato (top-100), refinar caro y
preciso (top-10), filtrar (umbral de score, dedup > 0.95, metadatos, presupuesto de tokens).

**MMR** (Carbonell & Goldstein, 1998): selección iterativa que equilibra relevancia y
novedad: `MMR(d) = λ·sim(q,d) − (1−λ)·max_{s∈S} sim(d,s)`. El re-ranking NO mejora el
recall: solo reordena lo que la 1.ª etapa trajo.

## 🧮 Ejemplo de referencia

MMR con `λ = 0.7`, k = 2. Scores: d1=0.90, d2=0.85, d3=0.70, d4=0.60;
sim(d1,d2)=0.95, sim(d1,d3)=0.30, sim(d1,d4)=0.20.

```text
Iter 1: S=∅ → gana d1 (0.90).
Iter 2: MMR(d2) = 0.7·0.85 − 0.3·0.95 = 0.310
        MMR(d3) = 0.7·0.70 − 0.3·0.30 = 0.400  ← gana
        MMR(d4) = 0.7·0.60 − 0.3·0.20 = 0.360
Selección: {d1, d3} — d2 queda fuera por casi-duplicado de d1.
```

Reproduce las tres multiplicaciones a mano: son la única aritmética que separa un
contexto redundante de uno diverso.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("retrieval", seed=101)
show(result)


## Reflexión

1. Si el recall@100 de la primera etapa es 0.60, ¿qué fracción máxima de respuestas correctas puede lograr el pipeline aunque el cross-encoder fuera perfecto, y por qué?
2. En el ejemplo MMR, ¿con qué valor aproximado de λ volvería d2 a entrar en la selección? Plantea la desigualdad `λ·0.85 − (1−λ)·0.95 > λ·0.70 − (1−λ)·0.30`.
3. ¿Por qué un umbral de score de cross-encoder calibrado en MS MARCO no es transferible a tu corpus interno, y qué datos necesitarías para recalibrarlo?